In [1]:
!pip install pandas
!pip install numpy
!pip install tensorflow



In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf

I0000 00:00:1773011548.838586   53785 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773011548.885543   53785 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773011550.221305   53785 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
a11=np.random.random(62500)*40-20
a12=np.random.random(62500)*40-20
a21=np.random.random(62500)*40-20
a22=np.random.random(62500)*40-20

b11=np.random.random(62500)*40-20
b12=np.random.random(62500)*40-20
b21=np.random.random(62500)*40-20
b22=np.random.random(62500)*40-20


df=pd.DataFrame({   'a11':a11,
                    'a12':a12,
                    'a21':a21,
                    'a22':a22,
                    'b11':b11,
                    'b12':b12,
                    'b21':b21,
                    'b22':b22,
                    })

print(len(df),len(df.drop_duplicates()))

df['R11']=(df['a11']*df['b11'])+(df['a12']*df['b21'])
df['R12']=(df['a11']*df['b12'])+(df['a12']*df['b22'])
df['R21']=(df['a21']*df['b11'])+(df['a22']*df['b21'])
df['R22']=(df['a21']*df['b12'])+(df['a22']*df['b22'])

df.head()

62500 62500


,a11,a12,a21,a22,b11,b12,b21,b22,R11,R12,R21,R22
0,14.823633,12.332931,-18.398789,10.386570,5.425406,11.267611,-17.110112,-11.853501,-130.593592,20.838521,-277.536280,-330.427615
1,-9.050358,11.558370,12.907410,6.671741,-1.764702,-17.480763,-9.846945,18.640205,-97.843447,373.657544,-88.474000,-101.268772
2,-17.956662,-17.088940,15.643419,12.822743,-3.868456,3.212940,-0.043583,6.282605,70.209346,-165.056737,-61.074734,130.821594
3,-11.758617,0.206207,15.915820,7.914989,-5.808730,13.859880,14.496768,3.106092,71.291971,-162.332525,22.291061,245.176047
4,0.006651,5.925254,-9.105407,-3.134909,-9.803840,0.639161,4.176481,17.845394,24.681511,105.742743,76.175064,-61.763501


In [4]:
df_train=df[:50000]
df_test=df[50000:]

print(len(df_train), len(df_test))

50000 12500


In [5]:
print(df.columns)

Index(['a11', 'a12', 'a21', 'a22', 'b11', 'b12', 'b21', 'b22', 'R11', 'R12',
       'R21', 'R22'],
      dtype='str')


In [6]:
salidas=['R11','R12','R21','R22']
entradas=['a11', 'a12', 'a21', 'a22', 'b11', 'b12', 'b21', 'b22']
tamaño=len(entradas)

x=[]

for i in entradas:
    x.append(df_train[i])

X=np.column_stack(x)

y=[]

for i in salidas:
    y.append(df_train[i])

Y=np.column_stack(y)

entrada = tf.keras.layers.Dense(units=tamaño, input_shape=[tamaño], activation="relu")
c1 = tf.keras.layers.Dense(units=128, activation="relu")
c2 = tf.keras.layers.Dense(units=128, activation="relu")
c3 = tf.keras.layers.Dense(units=128, activation="relu")
salida = tf.keras.layers.Dense(units=4, activation="linear")
red = tf.keras.Sequential([entrada, c1, c2, salida])
#red = tf.keras.Sequential([entrada, c1, c2,c3,c4, c5,c6, salida])
red.compile(optimizer='adam',
            loss="mse",
            metrics=['accuracy'])

/home/robotica/Jupyter_lab/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1773011552.249844   53785 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [7]:
historial = red.fit(X, Y, epochs=500, verbose=False)

In [ ]:
def clasificar_dataframe(df, modelo, columnas, salidas):
    X = df[columnas].values
    preds = modelo.predict(X)

    for i in range(len(salidas)):
        df[salidas[i]] = preds[:,i]
#    df['R11_a'] = preds[:,0]
    #df['Predeccion']=(preds > 0.5).astype(int)
    return df

salidas_2 = []
errores= []
for i in salidas:
    salidas_2.append(f'{i}_pred')
    errores.append(f'{i}_error')

clasificar_dataframe(df_test, red, entradas, salidas_2)

391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 904us/step


,a11,a12,a21,a22,b11,b12,b21,b22,R11,R12,R21,R22,R11_pred,R12_pred,R21_pred,R22_pred
50000,-10.296425,-16.269695,-11.134132,-16.911809,17.157695,-4.347177,16.513619,-12.578158,-445.334448,249.403177,-470.311200,261.121448,-457.202576,253.164154,-470.168945,265.480347
50001,-18.802763,-11.831320,-5.506835,-11.807522,13.249027,-1.524830,-6.659585,7.142879,-170.326624,-55.838673,5.672988,-75.942714,-170.501541,-55.422577,11.511721,-72.553711
50002,1.254182,15.613391,6.710497,-18.479639,6.164422,16.515157,5.620267,-3.079959,95.482728,-27.375588,-62.494160,167.741448,89.097206,-31.438814,-54.773144,168.938538
50003,7.003838,19.884678,-8.982899,15.441318,2.073978,6.859388,-6.084258,6.799289,-106.457710,183.243713,-112.579304,43.372789,-120.154640,179.928528,-114.976654,44.539490
50004,0.532905,2.122204,15.310877,-10.791114,-14.638088,-11.166432,-16.077932,15.291498,-41.921376,26.501031,-50.623173,-335.980172,-54.253853,22.650709,-54.647175,-321.974335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62495,16.452364,3.564549,-16.411788,3.980834,1.163320,18.996751,-4.151206,10.044708,4.342193,348.346312,-35.617432,-271.784347,-7.925302,349.181763,-31.825163,-276.846954
62496,-7.781946,8.954133,13.607717,-8.039546,-0.877521,10.283760,0.695623,18.804318,13.057524,88.348695,-17.533555,-11.239670,7.718307,79.295013,-15.273283,-7.656675
62497,-5.223015,-6.390902,-9.016899,-11.069522,-14.214023,-16.171646,-5.667530,3.800189,110.460682,60.178121,190.903250,103.751827,114.755043,57.699234,197.952499,107.614319
62498,-6.981147,-7.726787,18.707166,-6.997274,0.933717,16.828897,14.719697,15.027361,-120.254384,-233.598229,-85.530563,209.670399,-124.735313,-239.448364,-77.542961,215.318130


In [ ]:
df_test_

In [22]:

errores= []
for i in salidas:
    salidas_2.append(f'{i}_pred')
    errores.append(f'{i}_error')

for i in zip(salidas, salidas_2, errores):
    df_test[i[2]]=np.abs((df_test[i[0]]-df_test[i[1]])/df_test[i[0]])*100
    print(i)

df_test['error_media'] = df_test[errores].mean(axis=1)

df_test.head(50)

('R11', 'R11_pred', 'R11_error')
('R12', 'R12_pred', 'R12_error')
('R21', 'R21_pred', 'R21_error')
('R22', 'R22_pred', 'R22_error')


,a11,a12,a21,a22,b11,b12,b21,b22,R11,R12,...,R22,R11_pred,R12_pred,R21_pred,R22_pred,R11_error,R12_error,R21_error,R22_error,error_media
50000,-10.296425,-16.269695,-11.134132,-16.911809,17.157695,-4.347177,16.513619,-12.578158,-445.334448,249.403177,...,261.121448,-457.202576,253.164154,-470.168945,265.480347,2.664992,1.507991,0.030247,1.669299,1.468132
50001,-18.802763,-11.831320,-5.506835,-11.807522,13.249027,-1.524830,-6.659585,7.142879,-170.326624,-55.838673,...,-75.942714,-170.501541,-55.422577,11.511721,-72.553711,0.102695,0.745176,102.921640,4.462578,27.058022
50002,1.254182,15.613391,6.710497,-18.479639,6.164422,16.515157,5.620267,-3.079959,95.482728,-27.375588,...,167.741448,89.097206,-31.438814,-54.773144,168.938538,6.687620,14.842514,12.354780,0.713652,8.649642
50003,7.003838,19.884678,-8.982899,15.441318,2.073978,6.859388,-6.084258,6.799289,-106.457710,183.243713,...,43.372789,-120.154640,179.928528,-114.976654,44.539490,12.866076,1.809167,2.129476,2.689936,4.873664
50004,0.532905,2.122204,15.310877,-10.791114,-14.638088,-11.166432,-16.077932,15.291498,-41.921376,26.501031,...,-335.980172,-54.253853,22.650709,-54.647175,-321.974335,29.418112,14.528952,7.948932,4.168650,14.016161
50005,6.422542,-3.402325,-3.891404,6.567962,1.717307,-7.989230,-4.616258,-4.697179,26.735484,-35.329834,...,0.238428,22.076748,-36.669319,-36.269703,-1.671103,17.425291,3.791371,1.979448,800.883404,206.019879
50006,-1.073654,10.336212,10.511335,14.611446,-1.071100,5.076497,-11.013083,15.705734,-112.683572,156.887391,...,282.844232,-121.134819,157.288025,-178.154388,285.591431,7.499982,0.255364,3.472397,0.971276,3.049755
50007,19.767422,-7.142131,-1.989368,-1.883004,9.057282,-0.984606,-15.644792,-16.832062,290.776278,100.753671,...,33.653578,285.375885,102.770340,8.887758,38.131981,1.857233,2.001583,22.316161,13.307361,9.870585
50008,16.645617,-13.833755,19.374661,5.335015,15.456123,1.039224,-3.501191,-11.957636,305.711327,182.717530,...,-43.659549,306.592468,184.072296,285.759613,-43.874332,0.288227,0.741454,1.774130,0.491951,0.823940
50009,-8.676622,12.327230,-0.000014,-8.798785,9.089902,11.995731,15.472948,19.710406,111.868933,138.892272,...,-173.427799,106.783081,132.799423,-123.919640,-172.964294,4.546259,4.386744,8.978509,0.267261,4.544693
